<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS06/Praktikum4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install annoy faiss-cpu hnswlib

In [7]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 23.32881784439087 detik
Query time: 0.0003266334533691406 detik
Neighbors: [57442, 369591, 406820, 890823, 244484] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.008352994918823242 detik
Query time: 0.019053936004638672 detik
Neighbors: [ 57442 369591 406820 890823 244484] ...

=== HNSW (hnswlib) ===
Build time: 184.33038067817688 detik
Query time: 0.00025725364685058594 detik
Neighbors: [ 57442 369591 406820 890823 244484] ...


percobaan pada metric distance yang berbeda (Angular/Cosine)

In [8]:
# 1. Dataset 1 juta data 5D
np.random.seed(42)

n_data = 1_000_000
dim = 5

X = np.random.random((n_data, dim)).astype(np.float32)
query = np.random.random((1, dim)).astype(np.float32)
k = 10

results = []

# ===============================
# 2. Annoy (Angular)
# ===============================
print("=== Annoy (Angular) ===")

ann_index = AnnoyIndex(dim, "angular")

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
annoy_build = time.time() - start

start = time.time()
neighbors, distances = ann_index.get_nns_by_vector(
    query[0], k, include_distances=True
)
annoy_query = time.time() - start

print("Build time:", round(annoy_build, 4), "detik")
print("Query time:", round(annoy_query, 6), "detik")
print("Neighbors:", neighbors[:5])

results.append([
    "Angular/Cosine", "Annoy",
    round(annoy_build, 4),
    round(annoy_query, 6),
    neighbors[:5]
])

# ===============================
# 3. FAISS (Cosine Similarity)
# ===============================
print("\n=== FAISS (Cosine Similarity) ===")

X_faiss = X.copy()
query_faiss = query.copy()

# Normalisasi vektor agar inner product = cosine similarity
faiss.normalize_L2(X_faiss)
faiss.normalize_L2(query_faiss)

faiss_index = faiss.IndexFlatIP(dim)

start = time.time()
faiss_index.add(X_faiss)
faiss_build = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query_faiss, k)
faiss_query = time.time() - start

print("Build time:", round(faiss_build, 4), "detik")
print("Query time:", round(faiss_query, 6), "detik")
print("Neighbors:", indices[0][:5])

results.append([
    "Angular/Cosine", "FAISS",
    round(faiss_build, 4),
    round(faiss_query, 6),
    indices[0][:5].tolist()
])

# ===============================
# 4. HNSW (Cosine)
# ===============================
print("\n=== HNSW (Cosine) ===")

hnsw_index = hnswlib.Index(space="cosine", dim=dim)

start = time.time()
hnsw_index.init_index(
    max_elements=n_data,
    ef_construction=200,
    M=16
)
hnsw_index.add_items(X)
hnsw_build = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
hnsw_query = time.time() - start

print("Build time:", round(hnsw_build, 4), "detik")
print("Query time:", round(hnsw_query, 6), "detik")
print("Neighbors:", labels[0][:5])

results.append([
    "Angular/Cosine", "HNSW",
    round(hnsw_build, 4),
    round(hnsw_query, 6),
    labels[0][:5].tolist()
])

# ===============================
# 5. Tabel hasil percobaan
# ===============================
print("\nTABEL HASIL ANGULAR/COSINE")
print("=" * 100)

print(
    f"{'Metrik':<20}"
    f"{'Model':<12}"
    f"{'Build Time (s)':<18}"
    f"{'Query Time (s)':<18}"
    f"{'5 Nearest Neighbors'}"
)

print("-" * 100)

for row in results:
    print(
        f"{row[0]:<20}"
        f"{row[1]:<12}"
        f"{row[2]:<18}"
        f"{row[3]:<18}"
        f"{str(row[4])}"
    )

print("=" * 100)

=== Annoy (Angular) ===
Build time: 29.3597 detik
Query time: 0.000263 detik
Neighbors: [653774, 135588, 561017, 901095, 211473]

=== FAISS (Cosine Similarity) ===
Build time: 0.008 detik
Query time: 0.004047 detik
Neighbors: [653774 135588 561017 901095 211473]

=== HNSW (Cosine) ===
Build time: 175.921 detik
Query time: 0.000242 detik
Neighbors: [653774 135588 561017 901095 211473]

TABEL HASIL ANGULAR/COSINE
Metrik              Model       Build Time (s)    Query Time (s)    5 Nearest Neighbors
----------------------------------------------------------------------------------------------------
Angular/Cosine      Annoy       29.3597           0.000263          [653774, 135588, 561017, 901095, 211473]
Angular/Cosine      FAISS       0.008             0.004047          [653774, 135588, 561017, 901095, 211473]
Angular/Cosine      HNSW        175.921           0.000242          [653774, 135588, 561017, 901095, 211473]
